# 07. Machine Learning Models: Random Forest & XGBoost
### Implementation
- Supervised regression predicting strictly forward $k$-day realized volatility $RV_{t, t+k}$.
- Leakage-free features: return lags, historical volatilities, parkinson range, momentum, volume volatility.
- Hyperparameters tuned chronologically on past training/validation sets without lookahead bias.


In [ ]:
import sys
sys.path.insert(0, "..")
import pandas as pd
from src.features import build_feature_dataset
from src.ml_models import RandomForestVolatilityModel, XGBoostVolatilityModel
from src.validation import chronological_split

df = pd.read_csv("../data/processed/nifty50_daily_processed.csv", parse_dates=["Date"], index_col="Date")
feat_df = build_feature_dataset(df, target_horizon=5)

target_col = "target_rv_5d"
feature_cols = [c for c in feat_df.columns if c not in [target_col, "Close", "log_return", "simple_return"]]

train_df, val_df, test_df = chronological_split(feat_df, train_end="2018-12-31", val_end="2020-12-31")

rf = RandomForestVolatilityModel(n_estimators=200, max_depth=6).fit(train_df[feature_cols], train_df[target_col])
xgb = XGBoostVolatilityModel(n_estimators=200, max_depth=4, learning_rate=0.03).fit(train_df[feature_cols], train_df[target_col])

print("Top 5 RF Features:")
print(rf.get_feature_importances().head(5))
print("")
print("Top 5 XGBoost Features:")
print(xgb.get_feature_importances().head(5))
